In [1]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

pc = Pinecone(api_key=PINECONE_API_KEY) 

# (Commented out) IPython autoreload extension for Jupyter notebook development
# %load_ext autoreload
# %autoreload 2
# %reload_ext autoreload 

# Import custom modules
from backend.extract_filter import classify_and_extract
from backend.data_utils import fetch_papers, verify_results, chunk_papers, format_query, clear
from backend.Database import create_index, save_embeddings_to_index, delete_index
from backend.retrieval import answer_query, clear_memory, retrieve_chunks

from pinecone import Pinecone
import io
import contextlib

# Create and manage vector index
import pandas as pd
from urllib.error import HTTPError

# Importing async and evaluation tools
import asyncio
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import SemanticSimilarity
from ragas.embeddings import LangchainEmbeddingsWrapper

# Initialize ClearML for experiment tracking and artifact logging
import os
from dotenv import load_dotenv

load_dotenv()

CLEARML_WEB_HOST = os.getenv("CLEARML_WEB_HOST")
CLEARML_API_HOST = os.getenv("CLEARML_API_HOST")
CLEARML_FILES_HOST = os.getenv("CLEARML_FILES_HOST")
CLEARML_API_ACCESS_KEY = os.getenv("CLEARML_API_ACCESS_KEY")
CLEARML_API_SECRET_KEY = os.getenv("CLEARML_API_SECRET_KEY")

from clearml import Task, Logger
from time import sleep

'export' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


Pinecone API Key: [REDACTED]


c:\Users\guoji\Group-3\gjwenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

# from pinecone import Pinecone

# indexes = pc.list_indexes()

# # delete all indexes
# for idx in indexes:
#     index_name = idx["name"]
#     pc.delete_index(index_name)
#     print(f"Deleted index: {index_name}")



Deleted index: 1or3j6ef2a


In [3]:
# Create the index once (assuming it should persist during the conversation)
index_name = create_index()
index_obj = pc.Index(index_name)  

Index 'guonmb3s60' created successfully!


In [ ]:
clear_memory()
clear()

def handleUserQuery(user_input, user_index):
    # Use the provided user_input instead of the external variable
    classification = classify_and_extract(user_input)
    print(classification)
 
    filters = classification['filters']
    print("\nRunning Search with filters:", filters)

    # check if academic query is present
    if not format_query(filters):   
        question = classification['question']
        answer = answer_query(classification, index_obj, top_k=100)
        print("Generated Answer:")
        print(answer)
        return

    # Step 1: Fetch papers
    papers = fetch_papers(filters)

    # Step 2: Verify & Filter papers
    verified_papers = verify_results(papers, filters)
    print(f"[integration] Found {len(verified_papers)} verified paper(s).")
    
    # Determine number of papers to return; default to 1 if not provided
    try:
        num_papers = int(filters.get("max_results", 1))
    except ValueError:
        num_papers = 1
    num_papers = min(num_papers, len(verified_papers))

    # Step 3: Chunk papers
    chunks = chunk_papers(verified_papers[:num_papers])    # Get full paper content as chunks
    texts = [chunk["text"] for chunk in chunks]            # Extract just the text
    print("text:")
    print(texts)

    titles = [chunk["title"] for chunk in chunks]    # Extract the titles
    print('titles: ')
    print(titles)

    authors = [[author.lower() for author in chunk["author"]] for chunk in chunks]
    print("author:")
    print(authors)

    dates = [chunk["date"] for chunk in chunks]    # Extract the titles
    print('date: ')
    print(dates)


    if texts:
        # Save the full paper text chunks to the index in batches
        batch_size = 200
        for i in range(0, len(texts), batch_size): 
            print('i: ', i)
            text_batch = texts[i:i+batch_size]
            title_batch = titles[i:i+batch_size]     
            author_batch = authors[i:i+batch_size]    
            date_batch = dates[i:i+batch_size]         
            source_batch = ["arxiv"] * len(text_batch)
            save_embeddings_to_index(
                index_name=user_index, 
                text=text_batch, 
                titles=title_batch, 
                authors=author_batch, 
                dates=date_batch, 
                sources=source_batch,
                namespace="default"
            )
            sleep(1)



    #answer search question
    answer = answer_query(classification=classification, index_obj=index_obj, top_k=100)
    print("Generated Answer:")
    print(answer)
 
# Use a loop to enable continuous conversation
while True: 
    user_query = input("Enter your query (or type 'exit' to quit): ")

    if not user_query.strip():  # check if user input is empty
        print("Input cannot be empty. Please enter a valid query.")
        continue
    
    if user_query.lower() in ["exit", "quit"]:
        # delete_index(index_name=index_name)
        clear_memory()
        clear()
        break
    handleUserQuery(user_query, index_name)


Memory cleared.
{'action': 'other', 'filters': {'query': 'hello', 'author': None, 'title': None, 'category': None, 'abstract': None, 'journal_ref': None, 'doi': None, 'exclude_words': None, 'start_date': None, 'end_date': None, 'max_results': '1', 'sort_by': 'relevance'}, 'question': 'hello'}

Running Search with filters: {'query': 'hello', 'author': None, 'title': None, 'category': None, 'abstract': None, 'journal_ref': None, 'doi': None, 'exclude_words': None, 'start_date': None, 'end_date': None, 'max_results': '1', 'sort_by': 'relevance'}
added  The distributed Language Hello White Paper  to prevent searching it next time
[integration] Found 1 verified paper(s).
Extracted 266 chunks from 'The distributed Language Hello White Paper'

Total Chunks Extracted: 266

text:
['September 12, 2014 \n \nHello White Paper \n \n \n \nThe Distributed Language  \n \n \n \n \nV1.0 .3  (alpha ) \n \n \nWhite Paper  \nBoris Burshteyn \nbburshteyn@amsdec.com V1.0.3 (alpha) One Solution September 12, 

Token indices sequence length is longer than the specified maximum sequence length for this model (13877 > 1024). Running this sequence through the model will result in indexing errors
c:\Users\guoji\Group-3\backend\retrieval.py:219: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(
WARNING! top_p is not default parameter.
                    top_p was transferred to model_kwargs.
                    Please confirm that top_p is what you intended.
WARNING! frequency_penalty is not default parameter.
                    frequency_penalty was transferred to model_kwargs.
                    Please confirm that frequency_penalty is what you intended.
WARNING! presence_penalty i

Retrieved chunks: 
[{'text': '\'class HelloWorld {     // Broadcast to all known hosts \n    public static void main() { \n        hosts.+print("Hello, World!\\n" +  \n                            this_host.name() + ":-)\\n"); \n    }  \n}; \n \nThe two Hello concepts illustrated by this example are a\'\ntext from The distributed Language Hello White Paper\nauthored by boris burshteyn', 'title': 'the distributed language hello white paper', 'author': ['boris burshteyn'], 'date': 20140914.0, 'source': 'arxiv'}, {'text': "'transferring code, data and control flow between the \nengines from the connected hosts. Hello is a protocol -\nagnostic language, as it imposes no requirements on the \nunderlying network protocol; neither has it exposed any \nof the protocol elements to Hello programs3.'\ntext from The distributed Language Hello White Paper\nauthored by boris burshteyn", 'title': 'the distributed language hello white paper', 'author': ['boris burshteyn'], 'date': 20140914.0, 'source':

WARNING! top_p is not default parameter.
                    top_p was transferred to model_kwargs.
                    Please confirm that top_p is what you intended.
WARNING! frequency_penalty is not default parameter.
                    frequency_penalty was transferred to model_kwargs.
                    Please confirm that frequency_penalty is what you intended.
WARNING! presence_penalty is not default parameter.
                    presence_penalty was transferred to model_kwargs.
                    Please confirm that presence_penalty is what you intended.


Generated Answer:
content='content=\'Hey there! How can I help you today? If you have any questions or need information about the distributed language "Hello" or anything else, feel free to ask!\' additional_kwargs={} response_metadata={\'token_usage\': {\'completion_tokens\': 34, \'prompt_tokens\': 11290, \'total_tokens\': 11324, \'completion_tokens_details\': {\'accepted_prediction_tokens\': 0, \'audio_tokens\': 0, \'reasoning_tokens\': 0, \'rejected_prediction_tokens\': 0}, \'prompt_tokens_details\': {\'audio_tokens\': 0, \'cached_tokens\': 0}}, \'model_name\': \'gpt-4o-mini\', \'system_fingerprint\': \'fp_b376dfbbd5\', \'finish_reason\': \'stop\', \'logprobs\': None} id=\'run-7f8d2e16-9530-4c1a-a34e-ac26ba03f21c-1\'' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 197, 'prompt_tokens': 11461, 'total_tokens': 11658, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, '

BadRequestError: Error code: 400 - {'error': {'message': "'$.input' is invalid. Please check the API reference: https://platform.openai.com/docs/api-reference.", 'type': 'invalid_request_error', 'param': None, 'code': None}}

In [ ]:
# 